<a href="https://colab.research.google.com/github/aryanshenoy02/Banking-Fraud-Detection/blob/main/notebooks/isolation-forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving AryanTransactions.csv to AryanTransactions.csv


In [ ]:
import pandas as pd

try:
    # Skipping the first 3 rows of metadata to start directly from the table headers
    df = pd.read_csv('AryanTransactions.csv', skiprows=3)
    print("Successfully loaded by skipping the metadata rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading CSV: {e}")

Successfully loaded by skipping the metadata rows:


,Date,Time,Transaction Details,Transaction ID,UTR,Transaction Type,Credit/debit instrument,Amount
0,"Nov 07, 2026",10:19 pm,Paid to MYNTRA,T2611072219424673654963,4.455930e+11,DEBIT,Paid by 652926XXXXXXXX42,396.0
1,"Nov 07, 2026",12:20 pm,Paid to Gunavathi,T2611071220380263257149,1.816804e+11,DEBIT,Paid by 652926XXXXXXXX42,30.0
2,"Nov 03, 2026",04:46 pm,Paid to CHINCHU FEBIN,T2611031646116467551004,6.983029e+10,DEBIT,Paid by 652926XXXXXXXX42,210.0
3,"Nov 03, 2026",01:30 pm,Paid to Mr GOVIND RAJU,T2611031330551147580177,2.781873e+11,DEBIT,Paid by 652926XXXXXXXX42,42.0
4,"Oct 29, 2026",07:42 pm,Paid to Noor AMC,T2610291942338322376037,2.179925e+11,DEBIT,Paid by XXXX1629,325.0
5,"Oct 29, 2026",04:03 pm,Received from Lokesh,T2610291603030111345797,1.443723e+11,CREDIT,Credited to XXXX1629,55.0
6,"Oct 29, 2026",02:51 pm,Received from PRANATHI J,T2610291451591043848939,1.320329e+11,CREDIT,Credited to XXXX1629,50.0
7,"Oct 29, 2026",12:50 pm,Paid to Google Asia Pacific Pte.Ltd,T2610291249594816905913,2.306949e+11,DEBIT,Paid by 652926XXXXXXXX42,40.0
8,"Oct 28, 2026",06:08 pm,Paid to Sapna Book House -Royal Meenakshi Mall...,T2610281808346921321303,8.619950e+11,DEBIT,Paid by 652926XXXXXXXX42,100.0
9,"Oct 28, 2026",06:07 pm,Paid to Sapna Book House -Royal Meenakshi Mall...,T2610281806569657263913,7.960780e+11,DEBIT,Paid by 652926XXXXXXXX42,30.0


In [ ]:
combined = df['Date'].astype(str) + ' ' + df['Time'].astype(str)

parsed = pd.to_datetime(
    combined.str.replace(r'\bSept\b', 'Sep', regex=True),
    format='%b %d, %Y %I:%M %p',
    errors='coerce'
)

invalid_rows = df[parsed.isna()]

display(invalid_rows)
df = df[parsed.notna()].copy()

df['Datetime'] = parsed[parsed.notna()]

,Date,Time,Transaction Details,Transaction ID,UTR,Transaction Type,Credit/debit instrument,Amount
388,This is an automatically generated statement. ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
389,of any errors in the statement at https://supp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
390,terms-conditions/ for PhonePe Terms & Conditio...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
391,Disclaimer : Do not fall prey to fictitious of...,money circulation schemes and cheap funds,NaN,NaN,NaN,NaN,NaN,NaN
392,etc. through SMS,emails and calls. The contents of this email ...,NaN,NaN,NaN,NaN,NaN,NaN
393,the recipient specified in this document. If y...,please inform PhonePe at,NaN,NaN,NaN,NaN,NaN,NaN
394,https://support.phonepe.com/statement so that ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df['Date'] = df['Date'].str.replace(
    r'\bSept\b', 'Sep', regex=True
)

df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%b %d, %Y %I:%M %p'
)

In [ ]:
import numpy as np

# 1. Drop rows where all columns are NaN (if any exist at the end of the file)
df = df.dropna(how='all').copy()

# 2. Clean the 'Transaction Details' column
# We remove case-insensitive variations of "Paid to " and "Received from "
df['Transaction Details'] = df['Transaction Details'].str.replace(r'^(Paid to|Received from)\s+', '', case=False, regex=True)

# 3. Create a combined datetime column to sort chronologically and compute differences
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%b %d, %Y %I:%M %p')

# Sort chronologically so we can calculate the time difference forward in time
df = df.sort_values('Datetime').reset_index(drop=True)

# 4. Calculate the time difference between consecutive transactions (in hours)
df['Time_Diff_Hours'] = df['Datetime'].diff().dt.total_seconds() / 3600.0

# Display the cleaned and processed DataFrame
display(df[['Date', 'Time', 'Transaction Details', 'Transaction Type', 'Amount', 'Time_Diff_Hours']].head(10))

,Date,Time,Transaction Details,Transaction Type,Amount,Time_Diff_Hours
0,"Nov 12, 2025",09:22 am,KA01AR2800,DEBIT,25.00,NaN
1,"Nov 12, 2025",01:11 pm,JUBILANT FOODWORKS LTD,DEBIT,125.80,3.816667
2,"Nov 12, 2025",04:02 pm,Burger King,DEBIT,41.96,2.850000
3,"Nov 12, 2025",04:34 pm,Vighnesh (cheta) AMC,DEBIT,200.00,0.533333
4,"Nov 13, 2025",11:12 am,Reliance Tours & Travels,DEBIT,6.00,18.633333
5,"Nov 13, 2025",05:07 pm,Day to Day Super market,DEBIT,40.00,5.916667
6,"Nov 15, 2025",01:05 pm,ZEPTO MARKETPLACE PRIVATE LIMITED,DEBIT,124.00,43.966667
7,"Nov 17, 2025",06:07 pm,BEST CUTZ,DEBIT,100.00,53.033333
8,"Nov 18, 2025",07:40 am,BirthGiver(Amma),DEBIT,1.00,13.550000
9,"Nov 18, 2025",11:25 am,Bhavya refreshment,DEBIT,80.00,3.750000


In [ ]:
# sort to datatime
df = df.sort_values('Datetime').reset_index(drop=True)

# Calculate cumulative transaction count per counterparty (excluding current transaction)
df['Counterparty_Transaction_Count'] = df.groupby('Transaction Details').cumcount()

# Display the results emphasizing the new column
display(df[['Datetime', 'Transaction Details', 'Counterparty_Transaction_Count']].head(20))

,Datetime,Transaction Details,Counterparty_Transaction_Count
0,2025-11-12 09:22:00,KA01AR2800,0
1,2025-11-12 13:11:00,JUBILANT FOODWORKS LTD,0
2,2025-11-12 16:02:00,Burger King,0
3,2025-11-12 16:34:00,Vighnesh (cheta) AMC,0
4,2025-11-13 11:12:00,Reliance Tours & Travels,0
5,2025-11-13 17:07:00,Day to Day Super market,0
6,2025-11-15 13:05:00,ZEPTO MARKETPLACE PRIVATE LIMITED,0
7,2025-11-17 18:07:00,BEST CUTZ,0
8,2025-11-18 07:40:00,BirthGiver(Amma),0
9,2025-11-18 11:25:00,Bhavya refreshment,0


In [ ]:
import numpy as np

df['Log_Amount'] = np.log1p(df['Amount'])
df = df.drop(columns = ['Transaction ID', 'UTR', 'Date', 'Time'])
df.head()

,Transaction Details,Transaction Type,Credit/debit instrument,Amount,Datetime,Time_Diff_Hours,Counterparty_Transaction_Count,Log_Amount
0,KA01AR2800,DEBIT,Paid by UPI Lite,25.00,2025-11-12 09:22:00,NaN,0,3.258097
1,JUBILANT FOODWORKS LTD,DEBIT,Paid by UPI Lite,125.80,2025-11-12 13:11:00,3.816667,0,4.842611
2,Burger King,DEBIT,Paid by UPI Lite,41.96,2025-11-12 16:02:00,2.850000,0,3.760269
3,Vighnesh (cheta) AMC,DEBIT,Paid by UPI Lite,200.00,2025-11-12 16:34:00,0.533333,0,5.303305
4,Reliance Tours & Travels,DEBIT,Paid by UPI Lite,6.00,2025-11-13 11:12:00,18.633333,0,1.945910


In [ ]:
df.shape

(388, 8)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# 1. Prepare
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime').reset_index(drop=True)

df['hour'] = df['Datetime'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dayofweek'] = df['Datetime'].dt.dayofweek
df['is_new_counterparty'] = (df['Counterparty_Transaction_Count'] == 0).astype(int)
df['is_debit'] = (df['Transaction Type'] == 'DEBIT').astype(int)
#df = pd.get_dummies(df, columns=['Credit/debit instrument'], dtype=int)

# 2. Split by date
cut = int(len(df) * 0.75)
train = df.iloc[:cut].copy()
test = df.iloc[cut:].copy()

# 3. Choose features, fill NaN using TRAIN only
#inst_cols = [c for c in df.columns if c.startswith('Credit/debit instrument_')]
features = ['Log_Amount', 'Time_Diff_Hours', 'Counterparty_Transaction_Count',
            'is_new_counterparty', 'hour_sin', 'hour_cos', 'dayofweek',
            'is_debit']

median_gap = train['Time_Diff_Hours'].median()
train['Time_Diff_Hours'] = train['Time_Diff_Hours'].fillna(median_gap)
test['Time_Diff_Hours'] = test['Time_Diff_Hours'].fillna(median_gap)

# 4. Train and score
model = IsolationForest(n_estimators=200, random_state=42)
model.fit(train[features])

train_scores = model.decision_function(train[features])
test['score'] = model.decision_function(test[features])

# 5. Threshold and flag
threshold = np.percentile(train_scores, 1)
test['flagged'] = test['score'] < threshold

print(test['flagged'].mean())
print(test.sort_values('score').head(10)[['Transaction Details', 'Amount', 'Datetime', 'score']])

0.0
    Transaction Details   Amount            Datetime     score
384      Mr GOVIND RAJU    42.00 2026-11-03 13:30:00 -0.113333
295            NAGARAJU   253.00 2026-08-16 08:38:00 -0.108807
331         STAR BAZAAR   882.00 2026-09-16 19:50:00 -0.105159
291         STAR BAZAAR    45.00 2026-08-08 18:57:00 -0.102912
375       CHINCHU FEBIN   210.00 2026-10-25 12:40:00 -0.085619
376      Mr GOVIND RAJU    10.00 2026-10-27 10:51:00 -0.080463
302         STAR BAZAAR  1071.67 2026-08-20 12:06:00 -0.077242
311         STAR BAZAAR   719.00 2026-08-28 12:20:00 -0.076889
385       CHINCHU FEBIN   210.00 2026-11-03 16:46:00 -0.075929
344       CHINCHU FEBIN   210.00 2026-09-26 18:35:00 -0.074951


In [ ]:
df.head()

,Transaction Details,Transaction Type,Credit/debit instrument,Amount,Datetime,Time_Diff_Hours,Counterparty_Transaction_Count,Log_Amount,hour,hour_sin,hour_cos,dayofweek,is_new_counterparty,is_debit
0,KA01AR2800,DEBIT,Paid by UPI Lite,25.00,2025-11-12 09:22:00,NaN,0,3.258097,9,0.707107,-0.707107,2,1,1
1,JUBILANT FOODWORKS LTD,DEBIT,Paid by UPI Lite,125.80,2025-11-12 13:11:00,3.816667,0,4.842611,13,-0.258819,-0.965926,2,1,1
2,Burger King,DEBIT,Paid by UPI Lite,41.96,2025-11-12 16:02:00,2.850000,0,3.760269,16,-0.866025,-0.500000,2,1,1
3,Vighnesh (cheta) AMC,DEBIT,Paid by UPI Lite,200.00,2025-11-12 16:34:00,0.533333,0,5.303305,16,-0.866025,-0.500000,2,1,1
4,Reliance Tours & Travels,DEBIT,Paid by UPI Lite,6.00,2025-11-13 11:12:00,18.633333,0,1.945910,11,0.258819,-0.965926,3,1,1


In [ ]:
train.head()

,Transaction Details,Transaction Type,Credit/debit instrument,Amount,Datetime,Time_Diff_Hours,Counterparty_Transaction_Count,Log_Amount,hour,hour_sin,hour_cos,dayofweek,is_new_counterparty,is_debit
0,KA01AR2800,DEBIT,Paid by UPI Lite,25.00,2025-11-12 09:22:00,8.183333,0,3.258097,9,0.707107,-0.707107,2,1,1
1,JUBILANT FOODWORKS LTD,DEBIT,Paid by UPI Lite,125.80,2025-11-12 13:11:00,3.816667,0,4.842611,13,-0.258819,-0.965926,2,1,1
2,Burger King,DEBIT,Paid by UPI Lite,41.96,2025-11-12 16:02:00,2.850000,0,3.760269,16,-0.866025,-0.500000,2,1,1
3,Vighnesh (cheta) AMC,DEBIT,Paid by UPI Lite,200.00,2025-11-12 16:34:00,0.533333,0,5.303305,16,-0.866025,-0.500000,2,1,1
4,Reliance Tours & Travels,DEBIT,Paid by UPI Lite,6.00,2025-11-13 11:12:00,18.633333,0,1.945910,11,0.258819,-0.965926,3,1,1


In [ ]:
test.head()

,Transaction Details,Transaction Type,Credit/debit instrument,Amount,Datetime,Time_Diff_Hours,Counterparty_Transaction_Count,Log_Amount,hour,hour_sin,hour_cos,dayofweek,is_new_counterparty,is_debit,score,flagged
291,STAR BAZAAR,DEBIT,Paid by 652926XXXXXXXX42,45.0,2026-08-08 18:57:00,75.183333,21,3.828641,18,-1.000000,-1.836970e-16,5,0,1,-0.102912,False
292,CHINCHU FEBIN,DEBIT,Paid by 652926XXXXXXXX42,210.0,2026-08-11 13:34:00,66.616667,12,5.351858,13,-0.258819,-9.659258e-01,1,0,1,-0.017859,False
293,Day to day Super Market,DEBIT,Paid by 652926XXXXXXXX42,20.0,2026-08-11 20:27:00,6.883333,0,3.044522,20,-0.866025,5.000000e-01,1,1,1,0.051094,False
294,Day to day Super Market,DEBIT,Paid by 652926XXXXXXXX42,20.0,2026-08-11 20:27:00,0.000000,1,3.044522,20,-0.866025,5.000000e-01,1,0,1,0.024865,False
295,NAGARAJU,DEBIT,Paid by XXXX1629,253.0,2026-08-16 08:38:00,108.183333,0,5.537334,8,0.866025,-5.000000e-01,6,1,1,-0.108807,False
